Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

Dataset cleaning

In [7]:
train = pd.read_csv('C:/Users/iahas/OneDrive/Documents/New ML projects/COMP 6721/CUSTOMER RETENTION STRATEGIES IN E-COMMERCE/Train_Data.csv')
test  = pd.read_csv('C:/Users/iahas/OneDrive/Documents/New ML projects/COMP 6721/CUSTOMER RETENTION STRATEGIES IN E-COMMERCE/Test_Data.csv')

print('Raw shapes:', train.shape, test.shape)

print('\nStarting the dataset cleaning:')
# Checking for any missing values across all features in the training set
print('\nMissing values found in the training dataset:')
print(train.isnull().sum())

# We will impute missing numerical values using the median to ensure data consistency
num_cols = ['esent', 'eopenrate', 'eclickrate', 'avgorder', 'ordfreq']
for col in num_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col]  = test[col].fillna(median_val)

# Dropping the rows that are still missing dates
train = train.dropna(subset=['created', 'firstorder', 'lastorder'])
test  = test.dropna(subset=['created', 'firstorder', 'lastorder'])

# We will use the built in datetime found in pandas as it is easier to work with
date_cols = ['created', 'firstorder', 'lastorder']
for col in date_cols:
    train[col] = pd.to_datetime(train[col], dayfirst=True, format='mixed', errors='coerce')
    test[col]  = pd.to_datetime(test[col],  dayfirst=True, format='mixed', errors='coerce')

# We will identify any dates that failed to parse and were converted to NaT, and remove the collums
print('Unparseable dates turned into NaT:')
for col in date_cols:
    print(f'  train[{col}]: {train[col].isna().sum()} | test[{col}]: {test[col].isna().sum()}')
train = train.dropna(subset=date_cols)
test  = test.dropna(subset=date_cols)
print(f'\nShapes after dropping bad dates: train={train.shape}, test={test.shape}')

# Establishing a reference point to calculate the recency of customer activity
reference_date = train['lastorder'].max()

# We will engineer new features representing customer tenure and time since last purchase
for df in [train, test]:
    df['tenure_days']     = (df['lastorder']  - df['created']).dt.days
    df['days_to_first']   = (df['firstorder'] - df['created']).dt.days
    df['days_since_last'] = (reference_date   - df['lastorder']).dt.days

# Dropping the collumns the models do not need, we get:
drop_cols = ['custid', 'created', 'firstorder', 'lastorder']
train = train.drop(columns=drop_cols)
test  = test.drop(columns=drop_cols)

# Reviewing the statistical distribution of our cleaned features
print('\nStatistical distribution of our cleaned features')
print(train.describe().round(2))

print('\nFinal columns')
print(train.columns.tolist())
print('\nCleaning complete. Shapes:', train.shape, test.shape)

Raw shapes: (30801, 15) (38, 14)

Starting the dataset cleaning:

Missing values found in the training dataset:
custid        20
retained       0
created       20
firstorder    20
lastorder     20
esent          0
eopenrate      0
eclickrate     0
avgorder       0
ordfreq        0
paperless      0
refill         0
doorstep       0
favday         0
city           0
dtype: int64
Unparseable dates turned into NaT:
  train[created]: 0 | test[created]: 0
  train[firstorder]: 12 | test[firstorder]: 0
  train[lastorder]: 23 | test[lastorder]: 0

Shapes after dropping bad dates: train=(30758, 15), test=(38, 14)

Statistical distribution of our cleaned features
       retained     esent  eopenrate  eclickrate  avgorder   ordfreq  \
count  30758.00  30758.00   30758.00    30758.00  30758.00  30758.00   
mean       0.79     28.14      25.56        5.67     61.85      0.04   
std        0.40     16.75      29.56       10.57     40.95      0.10   
min        0.00      0.00       0.00        0.00   

K-Means